# 猫眼/大麦 座位图数据提取工具

## 使用步骤

1. 在 Chrome 中打开猫眼或大麦的演出页面（含座位图）
2. 打开 开发者工具 → Network，找到 `slots` 相关请求，复制 Response JSON 到 `raw_slots.json`
3. 或使用千岛内部 Chrome 插件一键导出 → 自动保存为 `slots_export.json`
4. 运行本脚本转换为 `theaters.js` 可直接使用的 `buildSections` 格式

---
**数据格式说明**：
- 猫眼 `slots` API 返回每个座位的 `rowNo`, `columnNo`, `areaName`, `seatNo`, `status` 等字段
- 大麦 类似字段但可能叫 `rowId`, `colId`, `areaLabel`

In [ ]:
import json
import re
from collections import defaultdict
from pathlib import Path

# ─── 配置 ────────────────────────────────────────────
SLOTS_FILE = 'slots_export.json'   # 插件导出文件路径
THEATER_ID = 'sh_example'          # 对应 theaters.js 中的 id
THEATER_NAME = '示例剧院'           # 剧院名称（调试用）
SOURCE = 'maoyan'                  # 'maoyan' 或 'damai'
# ─────────────────────────────────────────────────────

In [ ]:
# 加载原始 slots 数据
with open(SLOTS_FILE, 'r', encoding='utf-8') as f:
    raw = json.load(f)

print(f'原始数据 keys: {list(raw.keys()) if isinstance(raw, dict) else "list"}')
print(f'总记录数: {len(raw) if isinstance(raw, list) else "N/A"}')

In [ ]:
# ─── 标准化字段适配（猫眼 vs 大麦）───
def normalize_seat(seat_raw, source='maoyan'):
    """将不同平台的座位字段统一为标准格式"""
    if source == 'maoyan':
        return {
            'area':    seat_raw.get('areaName', seat_raw.get('standName', '未知区')),
            'area_id': seat_raw.get('areaId', ''),
            'row':     str(seat_raw.get('rowNo', seat_raw.get('row', ''))),
            'col':     int(seat_raw.get('columnNo', seat_raw.get('col', 0))),
            'seat_no': str(seat_raw.get('seatNo', seat_raw.get('columnNo', ''))),
            'status':  seat_raw.get('status', 1),  # 1=可售, 0=不可售/已售
        }
    elif source == 'damai':
        return {
            'area':    seat_raw.get('standName', seat_raw.get('areaLabel', '未知区')),
            'area_id': seat_raw.get('standId', ''),
            'row':     str(seat_raw.get('rowId', seat_raw.get('row', ''))),
            'col':     int(seat_raw.get('colId', seat_raw.get('col', 0))),
            'seat_no': str(seat_raw.get('seatLabel', seat_raw.get('colId', ''))),
            'status':  seat_raw.get('seatStatus', 1),
        }

# 提取 seats 列表（兼容多种数据结构）
if isinstance(raw, list):
    seats_raw = raw
elif 'slots' in raw:
    seats_raw = raw['slots']
elif 'data' in raw and 'slots' in raw['data']:
    seats_raw = raw['data']['slots']
elif 'data' in raw and isinstance(raw['data'], list):
    seats_raw = raw['data']
else:
    # 尝试递归查找 list 类型的值
    seats_raw = next((v for v in raw.values() if isinstance(v, list) and len(v) > 10), [])

seats = [normalize_seat(s, SOURCE) for s in seats_raw]
print(f'标准化后座位数: {len(seats)}')
if seats:
    print(f'示例: {seats[0]}')

In [ ]:
# ─── 按区段分组，统计行数 & 每行座位数 ───
from collections import OrderedDict

areas = defaultdict(lambda: defaultdict(list))  # area -> row -> [col]

for s in seats:
    areas[s['area']][s['row']].append(s['col'])

print(f'\n识别到 {len(areas)} 个区段：')
for area_name, rows in sorted(areas.items()):
    total_seats = sum(len(cols) for cols in rows.values())
    max_cols = max(len(cols) for cols in rows.values())
    print(f'  [{area_name}]  {len(rows)}排 × 最大{max_cols}座 = 约{total_seats}座')

In [ ]:
# ─── 生成 buildSections 配置 ───

# 区段类型映射（根据名称关键词判断）
def guess_section_type(name):
    name_lower = name.lower()
    if any(k in name for k in ['池座', '正厅', '一楼', '前区', '后区', 'stalls', 'orchestra']):
        return 'orchestra'
    if any(k in name for k in ['二楼', '包厢', 'VIP', '贵宾', 'mezzanine', 'circle']):
        return 'mezzanine'
    if any(k in name for k in ['三楼', '楼座', '顶层', '山顶', 'balcony', 'upper']):
        return 'balcony'
    if any(k in name for k in ['包厢', 'box', 'Box']):
        return 'box'
    return 'orchestra'

# 区段 ID 分配（A, B, C...）
section_configs = []
for idx, (area_name, rows) in enumerate(sorted(areas.items())):
    row_counts = [len(cols) for cols in rows.values()]
    max_cols = max(row_counts)
    mode_cols = max(set(row_counts), key=row_counts.count)  # 众数
    
    config = {
        'id': chr(65 + idx),  # A, B, C...
        'name': area_name,
        'type': guess_section_type(area_name),
        'rows': len(rows),
        'seatsPerRow': mode_cols,
        '_total': sum(row_counts),
    }
    section_configs.append(config)

print('生成的 buildSections 配置：\n')
for c in section_configs:
    print(f"  {{ id: '{c['id']}', name: '{c['name']}', type: '{c['type']}', rows: {c['rows']}, seatsPerRow: {c['seatsPerRow']} }},")

In [ ]:
# ─── 输出完整的 theaters.js 代码片段 ───

total_capacity = sum(c['_total'] for c in section_configs)

sections_code = '\n'.join(
    f"      {{ id: '{c['id']}', name: '{c['name']}', type: '{c['type']}', rows: {c['rows']}, seatsPerRow: {c['seatsPerRow']} }},"
    for c in section_configs
)

output = f"""
  // ─── 从猫眼/大麦爬取的真实座位数据 ───
  {{
    id: '{THEATER_ID}', name: '{THEATER_NAME}', cityId: 'shanghai',
    cover: 'TODO_替换为真实封面图URL',
    address: 'TODO_替换为地址', rating: 4.5, reviewCount: 0,
    tags: ['TODO'],
    description: 'TODO_替换为描述',
    transport: 'TODO', phone: 'TODO',
    halls: [{{ id: 'h1', name: '主剧场', capacity: {total_capacity}, sections: buildSections([
{sections_code}
    ])}}}],
  }},
"""

print(output)

# 同时保存到文件
out_path = Path(f'{THEATER_ID}_sections.js')
out_path.write_text(output, encoding='utf-8')
print(f'\n已保存到: {out_path.absolute()}')

In [ ]:
# ─── 可选：批量处理多个剧院 ───
# 将多个 slots_export_*.json 文件放在同一目录，批量转换

import os

BATCH_DIR = './'   # 包含多个 slots_*.json 的目录
OUTPUT_FILE = 'all_sections_output.js'

all_outputs = []

for fname in sorted(Path(BATCH_DIR).glob('slots_*.json')):
    theater_id = fname.stem.replace('slots_', '')
    print(f'处理: {fname.name} -> {theater_id}')
    
    try:
        with open(fname, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
        
        # 复用上面的逻辑（简化版）
        if isinstance(raw_data, list):
            s_raw = raw_data
        elif 'slots' in raw_data:
            s_raw = raw_data['slots']
        elif 'data' in raw_data:
            s_raw = raw_data['data'] if isinstance(raw_data['data'], list) else raw_data['data'].get('slots', [])
        else:
            s_raw = []
        
        s_seats = [normalize_seat(s, SOURCE) for s in s_raw]
        s_areas = defaultdict(lambda: defaultdict(list))
        for s in s_seats:
            s_areas[s['area']][s['row']].append(s['col'])
        
        s_configs = []
        for idx2, (aname, arows) in enumerate(sorted(s_areas.items())):
            row_cs = [len(c) for c in arows.values()]
            s_configs.append({
                'id': chr(65 + idx2),
                'name': aname,
                'type': guess_section_type(aname),
                'rows': len(arows),
                'seatsPerRow': max(set(row_cs), key=row_cs.count),
                '_total': sum(row_cs),
            })
        
        cap = sum(c['_total'] for c in s_configs)
        sec_code = '\n'.join(
            f"      {{ id: '{c['id']}', name: '{c['name']}', type: '{c['type']}', rows: {c['rows']}, seatsPerRow: {c['seatsPerRow']} }},"
            for c in s_configs
        )
        
        all_outputs.append(f"""
  // {theater_id} (从{fname.name}自动生成)
  {{
    id: '{theater_id}', name: 'TODO', cityId: 'shanghai',
    cover: 'TODO', address: 'TODO', rating: 4.5, reviewCount: 0,
    tags: ['TODO'], description: 'TODO', transport: 'TODO', phone: 'TODO',
    halls: [{{ id: 'h1', name: '主剧场', capacity: {cap}, sections: buildSections([
{sec_code}
    ])}}}],
  }},""")
        print(f'  OK: {len(s_seats)} 座位 / {len(s_areas)} 区段 / 约{cap}容量')
    except Exception as e:
        print(f'  ERROR: {e}')

if all_outputs:
    Path(OUTPUT_FILE).write_text('\n'.join(all_outputs), encoding='utf-8')
    print(f'\n批量输出已保存到: {OUTPUT_FILE}')
else:
    print('未找到 slots_*.json 文件，请将插件导出文件重命名为 slots_<theater_id>.json 格式')

## Chrome 插件使用说明

### 方法 A：Network 抓包（手动）
1. 打开 猫眼电影 → 演出列表 → 选座页面
2. `F12` 打开 DevTools → `Network` 标签 → 搜索 `slot`
3. 找到类似 `getSeatInfo` 或 `seatMap` 的请求
4. 点击 → `Response` → 复制全部 JSON 内容
5. 保存为 `slots_export.json`，放到本脚本同目录

### 方法 B：千岛内部 Chrome 插件（推荐）
1. 安装插件（见内部教程）
2. 打开猫眼演出选座页
3. 点击插件图标 → `导出座位图` → 自动下载 `slots_export.json`
4. 将文件移到本脚本目录，修改上面的 `SLOTS_FILE` 和 `THEATER_ID`
5. 运行所有 Cell

### 大麦网操作路径
1. 大麦 → 演出详情 → 立即购票 → 选座
2. Network 过滤 `seatmap` 或 `scene`
3. 找到含 `standName`/`rowId`/`colId` 的 JSON
4. 保存并设置 `SOURCE = 'damai'`

---
**输出文件**：`{THEATER_ID}_sections.js` 可直接复制到 `theaters.js` 的对应位置，
然后补充 `cover`/`address`/`description` 等字段即可。